# Performance

What CrocoDash's expensive steps cost on NCAR's machines, with fits for sizing jobs. Each
section says what was measured, gives the fit and what it means, and plots the measurements
(dots) against the fit (lines). The numbers are folded under each section.

The measurements are made by hand with pytest-benchmark and kept in `performance/*.json` (see
[How it works](how_it_works.md)). This page refits them every time the site is built.

In [ ]:
%config InlineBackend.figure_formats = ["svg"]
import math

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FuncFormatter, NullFormatter

import checks
from checks import fmt_gb, fmt_s

plt.rcParams.update(
    {
        "font.size": 9,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)


def plain_numbers(fig):
    """Label log axes 1, 10, 100, 1K, 1M, 1B rather than 10⁰, 10¹, 10²."""

    def label(v, _):
        for size, unit in ((1e9, "B"), (1e6, "M"), (1e3, "K")):
            if v >= size:
                return f"{v / size:g}{unit}"
        return f"{v:g}"

    for a in fig.axes:
        for axis, scale in ((a.xaxis, a.get_xscale()), (a.yaxis, a.get_yscale())):
            if scale == "log":
                axis.set_major_formatter(FuncFormatter(label))
                axis.set_minor_formatter(NullFormatter())


def fit(terms, measured):
    """Least-squares coefficients c with measured ≈ terms @ c (one row of terms per case)."""
    return np.linalg.lstsq(
        np.array(terms, float), np.array(measured, float), rcond=None
    )[0]


def fit_error(predicted, measured):
    """How far a fit is from the measurements, e.g. 'median 5%, worst 20%'."""
    error = np.abs(np.array(predicted) - np.array(measured)) / np.array(measured)
    return f"median {np.median(error):.0%}, worst {error.max():.0%}"


def cases(snapshot, test):
    """The cases one pytest-benchmark test ran, e.g. test_by_days."""
    return [b for b in snapshot["benchmarks"] if b["name"].startswith(test + "[")]


def info(b, key):
    """A value the benchmark recorded about its case (its extra_info)."""
    return b["extra_info"][key]


def seconds(b):
    return b["stats"]["mean"]

(obc)=
## Open boundaries

CrocoDash's open-boundary generation (`ConditionsConfigurator.process_bc`): regridding the
source ocean onto each boundary of the domain, then merging the pieces. The cases vary the
source resolution, the domain size, the number of days and the number of workers, on daily data
around a flat 1000 m box at 0.1° ([`test_obc.py`](https://github.com/CROCODILE-CESM/CrocoDashChecks/blob/main/performance/test_obc.py)).

:::{note}
The source is CrocoDash's synthetic `reference_ocean`, so nothing is downloaded and the download
isn't timed. It has 10 levels; GLORYS has 50.
:::

In [ ]:
obc = checks.load(checks.PERF / "obc.json")


def obc_terms(e):
    """One case's terms in the time fit and in the memory fit.

    REGRID splits the days into 30-day slices and hands min(workers, slices) forked
    workers an equal share each; a single worker runs in the main process."""
    S = e["slices"]  # 30-day slices
    k = math.ceil(S / e["workers"])  # slices per worker
    w = min(e["workers"], S) if e["workers"] > 1 else 0  # forked workers
    P = e["n_src"] / 1e6  # million source points
    C = e["n_cells"] / 1e6  # million domain cells
    time_terms = [1, S, S * C, k, k * P, k * C]
    memory_terms = [1, P, C, w, w * C]
    return time_terms, memory_terms


everything = obc["benchmarks"]
t = fit(
    [obc_terms(b["extra_info"])[0] for b in everything],
    [seconds(b) for b in everything],
)
m = fit(
    [obc_terms(b["extra_info"])[1] for b in everything],
    [info(b, "peak_mb") / 1024 for b in everything],
)


def obc_fit(e):
    """(seconds, GB) the fits give for a case."""
    time_terms, memory_terms = obc_terms(e)
    return float(np.dot(t, time_terms)), float(np.dot(m, memory_terms))


fitted = [obc_fit(b["extra_info"]) for b in everything]
checks.show(f"""
**The fit**, over all {len(everything)} cases:

- time (s) ≈ {t[0]:.2g} + S × ({t[1]:.2g} + {t[2]:.2g} C) + k × ({t[3]:.2g} + {t[4]:.2g} P + {t[5]:.2g} C)
- memory (GB) ≈ {m[0]:.2g} + {m[1]:.2g} P + {m[2]:.2g} C + w × ({m[3]:.2g} + {m[4]:.2g} C)

S is the number of 30-day slices, k slices per worker, w workers (0 for one), P million source
points and C million domain cells. Time {fit_error([f[0] for f in fitted], [seconds(b) for b in everything])};
memory {fit_error([f[1] for f in fitted], [info(b, "peak_mb") / 1024 for b in everything])}.
""")

In [ ]:
by_source = sorted(
    cases(obc, "test_by_source_resolution"), key=lambda b: info(b, "n_src")
)
by_size = sorted(cases(obc, "test_by_domain_size"), key=lambda b: info(b, "n_cells"))
by_days = sorted(cases(obc, "test_by_days"), key=lambda b: info(b, "days"))
by_workers = {}  # (box, source resolution) -> its cases, by worker count
for b in sorted(cases(obc, "test_by_workers"), key=lambda b: info(b, "workers")):
    by_workers.setdefault((info(b, "size_deg"), info(b, "source_res_deg")), []).append(
        b
    )

d0 = by_days[0]["extra_info"]
per_slice = obc_fit({**d0, "days": 60, "slices": 2})[0] - obc_fit(d0)[0]
speedups = [seconds(rows[0]) / seconds(rows[-1]) for rows in by_workers.values()]
most = max(info(b, "workers") for rows in by_workers.values() for b in rows)
checks.show(f"""
- **Source points:** linear, {t[4]:.0f} s more per million source points per 30 days (10° box, 1 worker).
- **Domain size:** linear too, {t[2] + t[5]:.0f} s more per million domain cells per 30 days (0.25° source, 1 worker).
- **Days:** linear, {per_slice:.1f} s per 30 days (10° box, 1 worker).
- **Workers:** time falls as 1/workers toward a floor: {min(speedups):.1f}–{max(speedups):.1f}× faster on
  {most} workers than on 1. Each worker takes whole slices, so with 12 slices, 6 to 11 workers all
  take as long as 6: use a worker count that divides the slices evenly.
- **Memory:** {m[2]:.1f} GB per million domain cells, plus {m[3]:.2f} GB per worker and {m[4]:.2f} GB
  per worker per million domain cells.
""")

fig, ax = plt.subplots(2, 3, figsize=(12, 7), layout="constrained")


def plot(ax, rows, x, y, fit_y, label=None):
    """Measured dots and the fit's line through the same cases."""
    xs = [x(b) for b in rows]
    (line,) = ax.plot(xs, [fit_y(b) for b in rows], "-", label=label)
    ax.plot(xs, [y(b) for b in rows], "o", color=line.get_color())


fit_s = lambda b: obc_fit(b["extra_info"])[0]
fit_gb = lambda b: obc_fit(b["extra_info"])[1]
peak_gb = lambda b: info(b, "peak_mb") / 1024

plot(ax[0, 0], by_source, lambda b: info(b, "n_src") / 1e6, seconds, fit_s)
ax[0, 0].set(
    title="Time by source points", xlabel="million source points (P)", ylabel="time (s)"
)
plot(ax[0, 1], by_size, lambda b: info(b, "n_cells") / 1e6, seconds, fit_s)
ax[0, 1].set(
    title="Time by domain size", xlabel="million domain cells (C)", ylabel="time (s)"
)
plot(ax[0, 2], by_days, lambda b: info(b, "days"), seconds, fit_s)
ax[0, 2].set(title="Time by days", xlabel="days", ylabel="time (s)")
for (box, source), rows in by_workers.items():
    label = f"{box:g}° box, {source:g}° source"
    workers = range(1, info(rows[-1], "workers") + 1)
    for a, measured, which in ((ax[1, 0], seconds, 0), (ax[1, 1], peak_gb, 1)):
        (line,) = a.plot(
            workers,
            [obc_fit({**rows[0]["extra_info"], "workers": w})[which] for w in workers],
            "-",
            label=label,
        )
        a.plot(
            [info(b, "workers") for b in rows],
            [measured(b) for b in rows],
            "o",
            color=line.get_color(),
        )
ax[1, 0].set(title="Time by workers (360 days)", xlabel="workers", ylabel="time (s)")
ax[1, 1].set(
    title="Peak memory by workers (360 days)",
    xlabel="workers",
    ylabel="peak memory (GB)",
)
ax[1, 0].legend()
plot(ax[1, 2], by_size, lambda b: info(b, "n_cells") / 1e6, peak_gb, fit_gb)
ax[1, 2].set(
    title="Peak memory by domain size",
    xlabel="million domain cells (C)",
    ylabel="peak memory (GB)",
)
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8, 3.6), layout="constrained")
for a, which, measured, unit in ((ax[0], 0, seconds, "s"), (ax[1], 1, peak_gb, "GB")):
    xs = [f[which] for f in fitted]
    ys = [measured(b) for b in everything]
    a.loglog(xs, ys, "o")
    lo, hi = min(xs + ys) / 1.5, max(xs + ys) * 1.5
    a.plot([lo, hi], [lo, hi], "-", color="grey", label="fit = measured")
    a.set(
        title=f"Fit against measurement: {['time', 'peak memory'][which]}",
        xlabel=f"fit ({unit})",
        ylabel=f"measured ({unit})",
    )
    a.legend()
plain_numbers(fig)
plt.show()

In [ ]:
def obc_table(rows, label):
    return checks.table(
        [label, "Source points", "Domain cells", "Time", "Fit", "Peak memory", "Fit"],
        [
            [
                name(b["extra_info"]),
                f"{info(b, 'n_src'):,}",
                f"{info(b, 'n_cells'):,}",
                fmt_s(seconds(b)),
                fmt_s(fit_s(b)),
                fmt_gb(peak_gb(b)),
                fmt_gb(fit_gb(b)),
            ]
            for b in rows
        ],
    )


text = ""
for title, rows, label, name in (
    (
        "By source resolution (10° box, 30 days, 1 worker)",
        by_source,
        "Source",
        lambda e: f"{e['source_res_deg']:g}°",
    ),
    (
        "By domain size (0.25° source, 30 days, 1 worker)",
        by_size,
        "Domain",
        lambda e: f"{e['size_deg']:g}° square",
    ),
    ("By days (10° box, 0.25° source, 1 worker)", by_days, "Days", lambda e: e["days"]),
    (
        "By workers (360 days)",
        [b for rows in by_workers.values() for b in rows],
        "Box, source, workers",
        lambda e: f"{e['size_deg']:g}°, {e['source_res_deg']:g}°, {e['workers']}",
    ),
):
    text += f"**{title}**\n\n{obc_table(rows, label)}\n\n"
checks.show(text)

(bathymetry)=
## Bathymetry

mom6_forge's `Topo.set_from_dataset`: GEBCO interpolated onto a 0.1° grid, for square domains
from 5° to 40° on a side ([`test_topo.py`](https://github.com/CROCODILE-CESM/CrocoDashChecks/blob/main/performance/test_topo.py)).
Memory is what the call added to the process.

In [ ]:
topo = sorted(
    cases(checks.load(checks.PERF / "topo.json"), "test_set_from_dataset"),
    key=lambda b: b["params"]["domain_deg"],
)
area = np.array([b["params"]["domain_deg"] ** 2 for b in topo])  # square degrees
topo_s = np.array([seconds(b) for b in topo])
topo_gb = np.array([info(b, "rss_mb") / 1024 for b in topo])


def power_fit(x, y):
    """(a, p) with y ≈ a·x^p, fitted as a straight line in log-log space."""
    p, log_a = np.polyfit(np.log(x), np.log(y), 1)
    return np.exp(log_a), p


(ta, tp), (ma, mp) = power_fit(area, topo_s), power_fit(area, topo_gb)
checks.show(f"""
**The fit**, over {len(topo)} cases (A is the domain's area in square degrees):

- time (s) ≈ {ta:.2g} · A{{sup}}`{tp:.2f}`
- memory added (GB) ≈ {ma:.2g} · A{{sup}}`{mp:.2f}`

Time {fit_error(ta * area**tp, topo_s)}; memory {fit_error(ma * area**mp, topo_gb)}.

- **Time** is close to linear in area: {fmt_s(ta * 1600**tp)} for a 40° square.
- **Memory** grows a bit faster than area: {fmt_gb(ma * 1600**mp)} for a 40° square.
""")

fig, ax = plt.subplots(1, 2, figsize=(8, 3.6), layout="constrained")
span = np.geomspace(area[0], area[-1], 40)
for a, y, (c, p), what in (
    (ax[0], topo_s, (ta, tp), "time (s)"),
    (ax[1], topo_gb, (ma, mp), "memory added (GB)"),
):
    (line,) = a.loglog(span, c * span**p, "-")
    a.loglog(area, y, "o", color=line.get_color())
    a.set(
        title=f"{what.split(' (')[0].capitalize()} by domain area",
        xlabel="square degrees (A)",
        ylabel=what,
    )
plain_numbers(fig)
plt.show()

In [ ]:
checks.show(
    checks.table(
        ["Domain", "Time", "Fit", "Memory added", "Fit"],
        [
            [
                f"{b['params']['domain_deg']}° square",
                fmt_s(s),
                fmt_s(ta * a**tp),
                fmt_gb(g),
                fmt_gb(ma * a**mp),
            ]
            for b, a, s, g in zip(topo, area, topo_s, topo_gb)
        ],
    )
)

(runoff)=
## Runoff mapping

mom6_forge's `gen_rof_maps`: the maps that carry GLOFAS's global river runoff to the ocean
grid, nearest neighbour and then smoothed over a radius rmax. The cases: a ladder of domains from
a 1° box to the Indo-Pacific, one domain at several resolutions, and a sweep over rmax
([`test_mapping.py`](https://github.com/CROCODILE-CESM/CrocoDashChecks/blob/main/performance/test_mapping.py)).
The smoothed map's weights are also what CESM's coupler holds in memory while the model runs.

In [ ]:
rof = checks.load(checks.PERF / "runoff_mapping.json")
rof_sets = {  # test: (title, a case's name)
    "test_gen_rof_maps_domain": (
        "By domain, rmax 20 km",
        lambda e: e["domain"].replace("_", " "),
    ),
    "test_gen_rof_maps_resolution": (
        "By resolution, Gulf of Mexico, rmax 20 km",
        lambda e: f"{e['resolution_deg']:.3g}°",
    ),
    "test_gen_rof_maps_rmax": (
        "By rmax, 1/12°",
        lambda e: f"{e['domain'].replace('_', ' ')}, rmax {e['rmax_km']} km",
    ),
}
rof_rows = {
    test: sorted(cases(rof, test), key=lambda b: (info(b, "n_ocn"), info(b, "rmax_km")))
    for test in rof_sets
}
rof_all = [b for rows in rof_rows.values() for b in rows]

# CESM's coupler holds the smoothed map as two route handles (liquid and ice runoff),
# ~167 bytes a weight each, measured on CrocIndoPacific_DROF_112 (4.0e8 weights,
# 133 GB on the mediator node).
COUPLER_BYTES_PER_WEIGHT = 2 * 167


def coupler_gb(weights):
    return weights * COUPLER_BYTES_PER_WEIGHT / 1024**3


def spread(e, rmax=None):
    """pi rmax^2 / a: about how many ocean cells lie within rmax of a coastal cell."""
    r = e["rmax_km"] if rmax is None else rmax
    return math.pi * r**2 / e["ocn_cell_km2"]


# Weights: n_s = N (1 + k spread), N the nearest-neighbour map's weights. An rmax
# narrower than a cell spreads nowhere (n_s = N), so those cases stay out of the fit.
wide = [b for b in rof_all if info(b, "rmax_km") ** 2 > info(b, "ocn_cell_km2")]
(k,) = fit(
    [[spread(b["extra_info"])] for b in wide],
    [info(b, "nnsm_n_s") / info(b, "nn_n_s") - 1 for b in wide],
)


def weights_fit(e, rmax=None):
    return e["nn_n_s"] * (1 + k * spread(e, rmax))


def cells_and_weights(b):
    """(million ocean cells, billion weights in the smoothed map)."""
    return info(b, "n_ocn") / 1e6, info(b, "nnsm_n_s") / 1e9


rof_s = [seconds(b) for b in rof_all]
rof_gb = [info(b, "peak_rss_mb") / 1024 for b in rof_all]
rt = fit([[1, *cells_and_weights(b)] for b in rof_all], rof_s)
rm = fit([[1, cells_and_weights(b)[1]] for b in rof_all], rof_gb)
rof_fit_s = (
    lambda b: rt[0] + rt[1] * cells_and_weights(b)[0] + rt[2] * cells_and_weights(b)[1]
)
rof_fit_gb = lambda b: rm[0] + rm[1] * cells_and_weights(b)[1]

# Weights against rmax: one exponent, shared by the domains.
rmax_rows = rof_rows["test_gen_rof_maps_rmax"]
domains = list(dict.fromkeys(info(b, "domain") for b in rmax_rows))
p = fit(
    [
        [float(info(b, "domain") == d) for d in domains]
        + [math.log(info(b, "rmax_km"))]
        for b in rmax_rows
    ],
    [math.log(info(b, "nnsm_n_s")) for b in rmax_rows],
)[-1]


def weights(n):
    return f"{n / 1e9:.2f} billion" if n >= 1e8 else f"{n / 1e6:.1f} million"


checks.show(f"""
**The fit**, over {len(rof_all)} cases:

- time (s) ≈ {rt[0]:.3g} + {rt[1]:.3g} C + {rt[2]:.3g} W
- memory (GB) ≈ {rm[0]:.3g} + {rm[1]:.3g} W
- weights ≈ N · (1 + {k:.2f} · π r² / a)
- coupler memory (GB) ≈ {coupler_gb(1e9):.0f} W

C is million ocean cells, W billion weights in the smoothed map, N the GLOFAS cells in the
domain's box (its area / 0.0025 deg²), r rmax (km) and a the ocean cells' area (km²). Time
{fit_error([rof_fit_s(b) for b in rof_all], rof_s)}; memory {fit_error([rof_fit_gb(b) for b in rof_all], rof_gb)};
weights {fit_error([weights_fit(b["extra_info"]) for b in wide], [info(b, "nnsm_n_s") for b in wide])}
(cases with rmax wider than a cell).

- **Time** is flat at {rt[0]:.0f} s, set by the global GLOFAS mesh, then {rt[1]:.0f} s more per
  million ocean cells.
- **Weights** go as about rmax{{sup}}`{p:.1f}`: doubling rmax multiplies the weights by {2**p:.1f}, and the
  coupler's memory with them.
- **Memory** is flat at {rm[0]:.0f} GB, set by the global GLOFAS mesh, then {rm[1]:.0f} GB more per
  billion weights.
""")

fig, ax = plt.subplots(1, 3, figsize=(12, 3.8), layout="constrained")
for test in ("test_gen_rof_maps_domain", "test_gen_rof_maps_resolution"):
    rows = rof_rows[test]
    xs = [info(b, "n_ocn") for b in rows]
    (line,) = ax[0].semilogx(
        xs, [rof_fit_s(b) for b in rows], "-", label=rof_sets[test][0]
    )
    ax[0].semilogx(xs, [seconds(b) for b in rows], "o", color=line.get_color())
ax[0].set(title="Time by ocean cells", xlabel="ocean cells", ylabel="time (s)")
ax[0].legend()

r_span = np.geomspace(
    min(info(b, "rmax_km") for b in rmax_rows),
    max(info(b, "rmax_km") for b in rmax_rows),
    40,
)
for d in domains:
    rows = [b for b in rmax_rows if info(b, "domain") == d]
    (line,) = ax[1].loglog(
        r_span,
        [weights_fit(rows[0]["extra_info"], r) for r in r_span],
        "-",
        label=d.replace("_", " "),
    )
    ax[1].loglog(
        [info(b, "rmax_km") for b in rows],
        [info(b, "nnsm_n_s") for b in rows],
        "o",
        color=line.get_color(),
    )
ax[1].set(title="Weights by rmax", xlabel="rmax (km)", ylabel="smoothed-map weights")
ax[1].set_xticks([10, 20, 40, 80])
ax[1].legend()

n_all = [info(b, "nnsm_n_s") for b in rof_all]
n_span = np.geomspace(min(n_all), max(n_all), 40)
(line,) = ax[2].semilogx(n_span, rm[0] + rm[1] * n_span / 1e9, "-")
ax[2].semilogx(n_all, rof_gb, "o", color=line.get_color())
ax[2].set(
    title="Peak memory by weights",
    xlabel="smoothed-map weights",
    ylabel="peak memory (GB)",
)
plain_numbers(fig)
plt.show()

In [ ]:
text = ""
for test, (title, name) in rof_sets.items():
    text += (
        f"**{title}**\n\n"
        + checks.table(
            [
                "Case",
                "Ocean cells",
                "Weights",
                "Fit",
                "Time",
                "Fit",
                "Peak memory",
                "Fit",
                "Coupler",
            ],
            [
                [
                    name(b["extra_info"]),
                    f"{info(b, 'n_ocn'):,}",
                    weights(info(b, "nnsm_n_s")),
                    weights(weights_fit(b["extra_info"])),
                    fmt_s(seconds(b)),
                    fmt_s(rof_fit_s(b)),
                    fmt_gb(info(b, "peak_rss_mb") / 1024),
                    fmt_gb(rof_fit_gb(b)),
                    fmt_gb(coupler_gb(info(b, "nnsm_n_s"))),
                ]
                for b in rof_rows[test]
            ],
        )
        + "\n\n"
    )
checks.show(text)

(mom6)=
## MOM6 speed vs. NTASKS_OCN

How MOM6's run time falls as it gets more MPI tasks, on 100×100×10 grids on Derecho: a sweep over
one Bahamas domain, and four more domains (the Caribbean, the Equatorial Pacific, the Arctic and
the Antarctic). These were run by hand; the scripts are in the CROC monorepo's
`dev/scaling_test/` and the results in
[`mom6_scaling.json`](https://github.com/CROCODILE-CESM/CrocoDashChecks/blob/main/performance/mom6_scaling.json).

In [ ]:
scaling = checks.load(checks.PERF / "mom6_scaling.json")
groups = {
    "Bahamas sweep": [scaling["reference_sweep"]],
    "Four domains": scaling["domains"],
}
runs = [
    (domain, point)
    for domains in groups.values()
    for domain in domains
    for point in domain["points"]
]
tasks = np.array([point["ntasks_ocn"] for _, point in runs])
run_s = np.array([point["tot_run_time_s"] for _, point in runs])
a, b = fit(
    np.column_stack([np.ones_like(tasks), 1 / tasks]), run_s
)  # run time = a + b / N
sweep = sorted(
    scaling["reference_sweep"]["points"], key=lambda point: point["ntasks_ocn"]
)

checks.show(f"""
**The fit**, over {len(runs)} runs ({scaling['compset']}, {scaling['date_range'][0]} to {scaling['date_range'][1]}):

- run time (s) ≈ {a:.3g} + {b:.4g} / N, with N = NTASKS_OCN

{fit_error(a + b / tasks, run_s)}. Run time falls as 1/N toward a {a:.0f} s floor:
{sweep[0]['tot_run_time_s'] / sweep[-1]['tot_run_time_s']:.0f}× faster on {sweep[-1]['ntasks_ocn']} tasks than on {sweep[0]['ntasks_ocn']}.
""")

fig, ax = plt.subplots(figsize=(5, 3.6), layout="constrained")
span = np.geomspace(tasks.min(), tasks.max(), 40)
ax.loglog(span, a + b / span, "-", color="grey", label="fit")
for name, domains in groups.items():
    points = [point for domain in domains for point in domain["points"]]
    ax.loglog(
        [q["ntasks_ocn"] for q in points],
        [q["tot_run_time_s"] for q in points],
        "o",
        label=name,
    )
ax.set(title="Run time by tasks", xlabel="NTASKS_OCN (N)", ylabel="run time (s)")
ax.legend()
plain_numbers(fig)
plt.show()

In [ ]:
checks.show(
    checks.table(
        ["Domain", "NTASKS_OCN", "Simulated years/day", "Run time", "Fit"],
        [
            [
                domain["name"],
                point["ntasks_ocn"],
                f"{point['throughput_sim_years_per_day']:.1f}",
                fmt_s(point["tot_run_time_s"]),
                fmt_s(a + b / point["ntasks_ocn"]),
            ]
            for domain, point in runs
        ],
    )
)